# **1) Carga el Dataset desde el repositorio de GitHub**

In [1]:
import os
import requests
import zipfile
import pandas as pd
import shutil

def descargar_y_desempacar_en_carpeta():

    # URL base del repositorio
    v_base_url = "https://raw.githubusercontent.com/jaramirezes/FaseII/main/datos/"
    v_archivo_zip = "imagenes_entrenamiento.zip"
    v_archivo_csv = "tabulador_costos_siniestros.csv"

    # 1. Validar conexión
    try:
        v_respuesta_repo = requests.get("https://github.com/jaramirezes/FaseII", timeout=10)
        if v_respuesta_repo.status_code == 200:
            print("Conexión exitosa a GitHub para descargar archivos del repositorio.\n")
        else:
            print(f"El repositorio devolvió HTTP {v_respuesta_repo.status_code}.\n")
            return
    except Exception as e:
        print(f"Error de red: {e}\n")
        return

    # 2. Descargar los archivos a /content
    print("Descargando archivos :")
    v_ruta_csv = f"/content/{v_archivo_csv}"
    v_ruta_zip = f"/content/{v_archivo_zip}"

    try:
        # CSV
        res_csv = requests.get(v_base_url + v_archivo_csv, timeout=15)
        if res_csv.status_code == 200:
            with open(v_ruta_csv, 'wb') as f: f.write(res_csv.content)
            print(f"\n Tabulador de costos : {v_ruta_csv}")
            df = pd.read_csv(v_ruta_csv)
            print(f" El CSV tiene {len(df)} registros.")

        # ZIP
        res_zip = requests.get(v_base_url + v_archivo_zip, timeout=60)
        if res_zip.status_code == 200:
            with open(v_ruta_zip, 'wb') as f: f.write(res_zip.content)
            print(f"\n Imagenes entrenamiento : {v_ruta_zip}\n")
    except Exception as e:
        print(f"  Al descargar: {e}")
        return

    # 3. Crear carpeta y desempacar el ZIP
    v_carpeta_destino = "/content/imagenes_entrenamiento"

    # Si la carpeta existe, la limpiamos para evitar duplicados
    if os.path.exists(v_carpeta_destino):
        shutil.rmtree(v_carpeta_destino)
    os.makedirs(v_carpeta_destino)

    print(f"Desempacando {v_archivo_zip} en {v_carpeta_destino}")
    try:
        with zipfile.ZipFile(v_ruta_zip, 'r') as v_zip:
            v_zip.extractall(v_carpeta_destino)
        print("  Desempacado exitoso.\n")

        # Eliminar el archivo ZIP después de desempacar para liberar espacio
        if os.path.exists(v_ruta_zip):
            os.remove(v_ruta_zip)
            print(f"Archivo ZIP eliminado correctamente: {v_ruta_zip}\n")

        # 4. Validación final
        v_elementos = os.listdir(v_carpeta_destino)
        print(f"La carpeta '{v_carpeta_destino}' contiene {len(v_elementos)} archivos.")

    except zipfile.BadZipFile:
        print(" El archivo no es un ZIP válido (es un puntero LFS). No se pudo desempacar.")
    except Exception as e:
        print(f" Error al desempacar: {e}")

if __name__ == "__main__":
    descargar_y_desempacar_en_carpeta()

Conexión exitosa a GitHub para descargar archivos del repositorio.

Descargando archivos :

 Tabulador de costos : /content/tabulador_costos_siniestros.csv
 El CSV tiene 390 registros.

 Imagenes entrenamiento : /content/imagenes_entrenamiento.zip

Desempacando imagenes_entrenamiento.zip en /content/imagenes_entrenamiento
  Desempacado exitoso.

Archivo ZIP eliminado correctamente: /content/imagenes_entrenamiento.zip

La carpeta '/content/imagenes_entrenamiento' contiene 780 archivos.


# **2) Entrenamiento**

# 2.1) Instalamos mlflow

In [2]:
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 79.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 92.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 73.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [mlflow]


# 2.2) Ejecutar entrenamiento

In [3]:
import os
import xml.etree.ElementTree as ET
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, AutoModelForImageClassification
from torch.optim import AdamW
import pandas as pd
from collections import Counter
import shutil
import mlflow

def cargar_y_entrenar_siniestros(directorio_modelo="./modelo_vit_siniestros"):
    print("Iniciando Procesamiento del Modelo por primera vez.")

    if os.path.exists(directorio_modelo):
        print(f"\nEliminando modelo anterior en '{directorio_modelo}' para reentrenamiento")
        shutil.rmtree(directorio_modelo)

    os.makedirs(directorio_modelo, exist_ok=True)
    print(f"Directorio '{directorio_modelo}' creado/limpiado exitosamente.")

    v_ruta_tabulador = "/content/tabulador_costos_siniestros.csv"
    v_directorio_extraccion = "/content/imagenes_entrenamiento"

    print(f"Archivo de tabulador identificado: {v_ruta_tabulador}")

    if os.path.exists(v_ruta_tabulador):
        try:
            if v_ruta_tabulador.endswith('.csv'):
                v_df_tabulador = pd.read_csv(v_ruta_tabulador)
            else:
                v_df_tabulador = pd.read_excel(v_ruta_tabulador)
            print(f"Tabulador de costos cargado exitosamente. Dimensiones: {v_df_tabulador.shape}")
        except Exception as e:
            print(f"Aviso al leer el tabulador: {e}")
    else:
        print(f"No se encontró el tabulador en la ruta especificada.")

    print(f"\nUsando carpeta de dataset en: {v_directorio_extraccion}")

    print("\nObteniendo clases dinamicas")
    v_clases_set = set()

    if os.path.exists(v_directorio_extraccion):
        for root, dirs, files_in_dir in os.walk(v_directorio_extraccion):
            for file in files_in_dir:
                if file.endswith('.xml'):
                    v_xml_path = os.path.join(root, file)
                    try:
                        v_tree = ET.parse(v_xml_path)
                        v_root_xml = v_tree.getroot()
                        for v_obj in v_root_xml.findall('object'):
                            v_nombre_clase_elem = v_obj.find('name')
                            if v_nombre_clase_elem is not None and v_nombre_clase_elem.text:
                                v_clases_set.add(v_nombre_clase_elem.text.strip())
                    except Exception as e:
                        pass

    v_clases_dano = sorted(list(v_clases_set))
    print(f"Clases detectadas ({len(v_clases_dano)}): {v_clases_dano}")

    if len(v_clases_dano) > 0:
        v_base_model = "google/vit-base-patch16-224"
        v_temp_processor = AutoImageProcessor.from_pretrained(v_base_model)

        v_id2label = {v_i: v_c for v_i, v_c in enumerate(v_clases_dano)}
        v_label2id = {v_c: v_i for v_i, v_c in enumerate(v_clases_dano)}

        v_temp_model = AutoModelForImageClassification.from_pretrained(
            v_base_model,
            num_labels=len(v_clases_dano),
            id2label=v_id2label,
            label2id=v_label2id,
            ignore_mismatched_sizes=True
        )
        v_temp_processor.save_pretrained(directorio_modelo)
        v_temp_model.save_pretrained(directorio_modelo)

    class DatasetSiniestros(Dataset):
        def __init__(self, directorio_raiz, image_processor, clases_disponibles):
            self.directorio_raiz = directorio_raiz
            self.image_processor = image_processor
            self.clases_disponibles = clases_disponibles
            self.muestras = []
            v_conteo_clases = []

            if os.path.exists(directorio_raiz):
                for root, dirs, files_in_dir in os.walk(directorio_raiz):
                    for file in files_in_dir:
                        if file.endswith(('.jpg', '.jpeg', '.png')):
                            img_path = os.path.join(root, file)
                            xml_path = os.path.splitext(img_path)[0] + ".xml"
                            clase_idx = 0

                            if os.path.exists(xml_path):
                                try:
                                    tree = ET.parse(xml_path)
                                    root_xml = tree.getroot()
                                    obj = root_xml.find('object')
                                    if obj is not None:
                                        nombre_clase_elem = obj.find('name')
                                        if nombre_clase_elem is not None and nombre_clase_elem.text:
                                            txt_clase = nombre_clase_elem.text.strip()
                                            if txt_clase in self.clases_disponibles:
                                                clase_idx = self.clases_disponibles.index(txt_clase)
                                except Exception as e:
                                    pass

                            self.muestras.append((img_path, clase_idx))
                            v_conteo_clases.append(clase_idx)

            print("\nDistribución de clases detectadas en XML")
            v_distribucion = Counter(v_conteo_clases)
            for idx, count in v_distribucion.items():
                print(f"   [{idx}] {self.clases_disponibles[idx]}: {count} imágenes")

        def __len__(self):
            return len(self.muestras)

        def __getitem__(self, idx):
            img_path, label = self.muestras[idx]
            image = Image.open(img_path).convert("RGB")
            inputs = self.image_processor(images=image, return_tensors="pt")
            pixel_values = inputs["pixel_values"].squeeze(0)
            return pixel_values, torch.tensor(label, dtype=torch.long)

    if os.path.exists(directorio_modelo):
        v_image_processor = AutoImageProcessor.from_pretrained(directorio_modelo, local_files_only=True)
    else:
        v_image_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")

    v_dataset = DatasetSiniestros(v_directorio_extraccion, v_image_processor, v_clases_dano)

    if len(v_dataset) > 0:
        v_dataloader = DataLoader(v_dataset, batch_size=4, shuffle=True)
        print(f"\nTotal de imágenes listas para entrenamiento: {len(v_dataset)}")

        v_model = AutoModelForImageClassification.from_pretrained(
            directorio_modelo, num_labels=len(v_clases_dano), ignore_mismatched_sizes=True, local_files_only=True
        )

        v_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Entrenando en dispositivo: {v_device}")
        v_model.to(v_device)
        v_model.train()

        v_learning_rate = 2e-5
        v_epocas = 2
        v_batch_size = 4

        v_optimizer = AdamW(v_model.parameters(), lr=v_learning_rate)

        mlflow.start_run()
        mlflow.log_params({
            "learning_rate": v_learning_rate,
            "epocas": v_epocas,
            "batch_size": v_batch_size,
            "num_classes": len(v_clases_dano)
        })

        print("\nIniciando ciclo de entrenamiento (2 épocas)")
        for epoca in range(v_epocas):
            v_suma_loss = 0.0
            for pixel_values, labels in v_dataloader:
                pixel_values, labels = pixel_values.to(v_device), labels.to(v_device)
                v_optimizer.zero_grad()
                v_outputs = v_model(pixel_values=pixel_values, labels=labels)
                v_loss = v_outputs.loss
                v_loss.backward()
                v_optimizer.step()
                v_suma_loss += v_loss.item()

            v_loss_promedio = v_suma_loss / len(v_dataloader)
            print(f"Época {epoca+1}/{v_epocas} completada - Loss promedio: {v_loss_promedio:.4f}")
            mlflow.log_metric("loss", v_loss_promedio, step=epoca)

        mlflow.end_run()

        v_model.save_pretrained(directorio_modelo)
        v_image_processor.save_pretrained(directorio_modelo)
        print(f"\nEntrenamiento finalizado, Modelo guardado en: {directorio_modelo}")
    else:
        print("No se encontraron imágenes válidas para entrenar.")

if __name__ == "__main__":
    cargar_y_entrenar_siniestros()

Iniciando Procesamiento del Modelo por primera vez.
Directorio './modelo_vit_siniestros' creado/limpiado exitosamente.
Archivo de tabulador identificado: /content/tabulador_costos_siniestros.csv
Tabulador de costos cargado exitosamente. Dimensiones: (390, 7)

Usando carpeta de dataset en: /content/imagenes_entrenamiento

Obteniendo clases dinamicas
Clases detectadas (15): ['Front-Windscreen-Damage', 'Headlight-Damage', 'Major-Rear-Bumper-Dent', 'Rear-windscreen-Damage', 'RunningBoard-Dent', 'Sidemirror-Damage', 'Signlight-Damage', 'Taillight-Damage', 'bonnet-dent', 'doorouter-dent', 'fender-dent', 'front-bumper-dent', 'quaterpanel-dent', 'rear-bumper-dent', 'roof-dent']


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

[transformers] You passed `num_labels=15` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

2026/11/16 01:50:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/16 01:50:28 INFO mlflow.store.db.utils: Updating database tables



Iniciando ciclo de entrenamiento (2 épocas)
Época 1/2 completada - Loss promedio: 2.2965
Época 2/2 completada - Loss promedio: 1.0923


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Entrenamiento finalizado, Modelo guardado en: ./modelo_vit_siniestros


2.3) Ejecutamos la consulta a MLFlow

In [6]:
import mlflow
v_runs = mlflow.search_runs()
display(v_runs)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.loss,params.learning_rate,params.epocas,params.batch_size,params.num_classes,tags.mlflow.source.name,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.user
0,46c324138981469a952fb6c343961824,0,FINISHED,/content/mlruns/0/46c324138981469a952fb6c34396...,2026-11-16 01:50:31.886000+00:00,2026-11-16 01:51:07.071000+00:00,1.092291,2e-05,2,4,15,fileId=13Q8hFXQDLEkjjlAk_5jwjnxbJfJAFEsJ,NOTEBOOK,victorious-dog-712,root


# **3) - Ejecutamos nuevamente el modelo y agregamos las metricas de Accuracy, Precision, Recall, F1 Score, Tiempo de entrenamiento**

In [7]:
import os
import xml.etree.ElementTree as ET
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, AutoModelForImageClassification
from torch.optim import AdamW
import pandas as pd
from collections import Counter
import shutil
import mlflow
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def cargar_y_entrenar_siniestros(directorio_modelo="./modelo_vit_siniestros"):
    print("Iniciando Procesamiento del Modelo.")

    if os.path.exists(directorio_modelo):
        print(f"\nEliminando modelo anterior en '{directorio_modelo}' para reentrenamiento")
        shutil.rmtree(directorio_modelo)

    os.makedirs(directorio_modelo, exist_ok=True)
    print(f"Directorio '{directorio_modelo}' creado/limpiado exitosamente.")

    v_ruta_tabulador = "/content/tabulador_costos_siniestros.csv"
    v_directorio_extraccion = "/content/imagenes_entrenamiento"

    print(f"Archivo de tabulador identificado: {v_ruta_tabulador}")

    if os.path.exists(v_ruta_tabulador):
        try:
            if v_ruta_tabulador.endswith('.csv'):
                v_df_tabulador = pd.read_csv(v_ruta_tabulador)
            else:
                v_df_tabulador = pd.read_excel(v_ruta_tabulador)
            print(f"Tabulador de costos cargado exitosamente. Dimensiones: {v_df_tabulador.shape}")
        except Exception as e:
            print(f"Aviso al leer el tabulador: {e}")
    else:
        print(f"No se encontró el tabulador en la ruta especificada.")

    print(f"\nUsando carpeta de dataset en: {v_directorio_extraccion}")

    print("\nObteniendo clases dinamicas")
    v_clases_set = set()

    if os.path.exists(v_directorio_extraccion):
        for root, dirs, files_in_dir in os.walk(v_directorio_extraccion):
            for file in files_in_dir:
                if file.endswith('.xml'):
                    v_xml_path = os.path.join(root, file)
                    try:
                        v_tree = ET.parse(v_xml_path)
                        v_root_xml = v_tree.getroot()
                        for v_obj in v_root_xml.findall('object'):
                            v_nombre_clase_elem = v_obj.find('name')
                            if v_nombre_clase_elem is not None and v_nombre_clase_elem.text:
                                v_clases_set.add(v_nombre_clase_elem.text.strip())
                    except Exception as e:
                        pass

    v_clases_dano = sorted(list(v_clases_set))
    print(f"Clases detectadas ({len(v_clases_dano)}): {v_clases_dano}")

    if len(v_clases_dano) > 0:
        v_base_model = "google/vit-base-patch16-224"
        v_temp_processor = AutoImageProcessor.from_pretrained(v_base_model)

        v_id2label = {v_i: v_c for v_i, v_c in enumerate(v_clases_dano)}
        v_label2id = {v_c: v_i for v_i, v_c in enumerate(v_clases_dano)}

        v_temp_model = AutoModelForImageClassification.from_pretrained(
            v_base_model,
            num_labels=len(v_clases_dano),
            id2label=v_id2label,
            label2id=v_label2id,
            ignore_mismatched_sizes=True
        )
        v_temp_processor.save_pretrained(directorio_modelo)
        v_temp_model.save_pretrained(directorio_modelo)

    class DatasetSiniestros(Dataset):
        def __init__(self, directorio_raiz, image_processor, clases_disponibles):
            self.directorio_raiz = directorio_raiz
            self.image_processor = image_processor
            self.clases_disponibles = clases_disponibles
            self.muestras = []
            v_conteo_clases = []

            if os.path.exists(directorio_raiz):
                for root, dirs, files_in_dir in os.walk(directorio_raiz):
                    for file in files_in_dir:
                        if file.endswith(('.jpg', '.jpeg', '.png')):
                            img_path = os.path.join(root, file)
                            xml_path = os.path.splitext(img_path)[0] + ".xml"
                            clase_idx = 0

                            if os.path.exists(xml_path):
                                try:
                                    tree = ET.parse(xml_path)
                                    root_xml = tree.getroot()
                                    obj = root_xml.find('object')
                                    if obj is not None:
                                        nombre_clase_elem = obj.find('name')
                                        if nombre_clase_elem is not None and nombre_clase_elem.text:
                                            txt_clase = nombre_clase_elem.text.strip()
                                            if txt_clase in self.clases_disponibles:
                                                clase_idx = self.clases_disponibles.index(txt_clase)
                                except Exception as e:
                                    pass

                            self.muestras.append((img_path, clase_idx))
                            v_conteo_clases.append(clase_idx)

            print("\nDistribución de clases detectadas en XML")
            v_distribucion = Counter(v_conteo_clases)
            for idx, count in v_distribucion.items():
                print(f"   [{idx}] {self.clases_disponibles[idx]}: {count} imágenes")

        def __len__(self):
            return len(self.muestras)

        def __getitem__(self, idx):
            img_path, label = self.muestras[idx]
            image = Image.open(img_path).convert("RGB")
            inputs = self.image_processor(images=image, return_tensors="pt")
            pixel_values = inputs["pixel_values"].squeeze(0)
            return pixel_values, torch.tensor(label, dtype=torch.long)

    if os.path.exists(directorio_modelo):
        v_image_processor = AutoImageProcessor.from_pretrained(directorio_modelo, local_files_only=True)
    else:
        v_image_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")

    v_dataset = DatasetSiniestros(v_directorio_extraccion, v_image_processor, v_clases_dano)

    if len(v_dataset) > 0:
        v_dataloader = DataLoader(v_dataset, batch_size=4, shuffle=True)
        print(f"\nTotal de imágenes listas para entrenamiento: {len(v_dataset)}")

        v_model = AutoModelForImageClassification.from_pretrained(
            directorio_modelo, num_labels=len(v_clases_dano), ignore_mismatched_sizes=True, local_files_only=True
        )

        v_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Entrenando en dispositivo: {v_device}")
        v_model.to(v_device)
        v_model.train()

        v_learning_rate = 2e-5
        v_epocas = 2
        v_batch_size = 4

        v_optimizer = AdamW(v_model.parameters(), lr=v_learning_rate)

        mlflow.start_run()
        mlflow.log_params({
            "learning_rate": v_learning_rate,
            "epocas": v_epocas,
            "batch_size": v_batch_size,
            "num_classes": len(v_clases_dano)
        })

        v_tiempo_inicio = time.time()

        print("\nIniciando ciclo de entrenamiento (2 épocas)")
        for epoca in range(v_epocas):
            v_suma_loss = 0.0
            v_y_true = []
            v_y_pred = []

            for pixel_values, labels in v_dataloader:
                pixel_values, labels = pixel_values.to(v_device), labels.to(v_device)
                v_optimizer.zero_grad()
                v_outputs = v_model(pixel_values=pixel_values, labels=labels)
                v_loss = v_outputs.loss
                v_loss.backward()
                v_optimizer.step()
                v_suma_loss += v_loss.item()

                v_preds = torch.argmax(v_outputs.logits, dim=1)
                v_y_true.extend(labels.cpu().numpy())
                v_y_pred.extend(v_preds.cpu().numpy())

            v_loss_promedio = v_suma_loss / len(v_dataloader)
            v_accuracy = accuracy_score(v_y_true, v_y_pred)
            v_precision = precision_score(v_y_true, v_y_pred, average='weighted', zero_division=0)
            v_recall = recall_score(v_y_true, v_y_pred, average='weighted', zero_division=0)
            v_f1 = f1_score(v_y_true, v_y_pred, average='weighted', zero_division=0)

            print(f"Época {epoca+1}/{v_epocas} - Loss: {v_loss_promedio:.4f} | Acc: {v_accuracy:.4f} | F1: {v_f1:.4f}")
            mlflow.log_metrics({
                "loss": v_loss_promedio,
                "accuracy": v_accuracy,
                "precision": v_precision,
                "recall": v_recall,
                "f1_score": v_f1
            }, step=epoca)

        v_tiempo_entrenamiento = time.time() - v_tiempo_inicio
        mlflow.log_metric("tiempo_entrenamiento_seg", v_tiempo_entrenamiento)
        print(f"Tiempo total de entrenamiento: {v_tiempo_entrenamiento:.2f} segundos")

        mlflow.end_run()

        v_model.save_pretrained(directorio_modelo)
        v_image_processor.save_pretrained(directorio_modelo)
        print(f"\nEntrenamiento finalizado, Modelo guardado en: {directorio_modelo}")
    else:
        print("No se encontraron imágenes válidas para entrenar.")

if __name__ == "__main__":
    cargar_y_entrenar_siniestros()

Iniciando Procesamiento del Modelo.

Eliminando modelo anterior en './modelo_vit_siniestros' para reentrenamiento
Directorio './modelo_vit_siniestros' creado/limpiado exitosamente.
Archivo de tabulador identificado: /content/tabulador_costos_siniestros.csv
Tabulador de costos cargado exitosamente. Dimensiones: (390, 7)

Usando carpeta de dataset en: /content/imagenes_entrenamiento

Obteniendo clases dinamicas
Clases detectadas (15): ['Front-Windscreen-Damage', 'Headlight-Damage', 'Major-Rear-Bumper-Dent', 'Rear-windscreen-Damage', 'RunningBoard-Dent', 'Sidemirror-Damage', 'Signlight-Damage', 'Taillight-Damage', 'bonnet-dent', 'doorouter-dent', 'fender-dent', 'front-bumper-dent', 'quaterpanel-dent', 'rear-bumper-dent', 'roof-dent']


[transformers] You passed `num_labels=15` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([15])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([15, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Distribución de clases detectadas en XML
   [1] Headlight-Damage: 24 imágenes
   [13] rear-bumper-dent: 49 imágenes
   [10] fender-dent: 28 imágenes
   [11] front-bumper-dent: 71 imágenes
   [12] quaterpanel-dent: 27 imágenes
   [4] RunningBoard-Dent: 12 imágenes
   [8] bonnet-dent: 50 imágenes
   [9] doorouter-dent: 35 imágenes
   [0] Front-Windscreen-Damage: 15 imágenes
   [3] Rear-windscreen-Damage: 27 imágenes
   [7] Taillight-Damage: 20 imágenes
   [5] Sidemirror-Damage: 19 imágenes
   [14] roof-dent: 11 imágenes
   [6] Signlight-Damage: 2 imágenes

Total de imágenes listas para entrenamiento: 390


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Entrenando en dispositivo: cuda

Iniciando ciclo de entrenamiento (2 épocas)
Época 1/2 - Loss: 2.2163 | Acc: 0.3154 | F1: 0.2711
Época 2/2 - Loss: 1.0203 | Acc: 0.8154 | F1: 0.8018
Tiempo total de entrenamiento: 35.35 segundos


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Entrenamiento finalizado, Modelo guardado en: ./modelo_vit_siniestros


# 3.1) Ejecución a consulta MLFlow

In [10]:
import mlflow
v_runs = mlflow.search_runs()
display(v_runs)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.f1_score,metrics.accuracy,metrics.tiempo_entrenamiento_seg,metrics.loss,metrics.precision,metrics.recall,params.learning_rate,params.epocas,params.batch_size,params.num_classes,tags.mlflow.source.name,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.user
0,9fbbacbea436461c98f704623d7e9882,0,FINISHED,/content/mlruns/0/9fbbacbea436461c98f704623d7e...,2026-11-16 01:59:54.604000+00:00,2026-11-16 02:00:30.442000+00:00,0.801753,0.815385,35.346145,1.020296,0.833076,0.815385,2e-05,2,4,15,fileId=13Q8hFXQDLEkjjlAk_5jwjnxbJfJAFEsJ,NOTEBOOK,awesome-sloth-148,root
1,46c324138981469a952fb6c343961824,0,FINISHED,/content/mlruns/0/46c324138981469a952fb6c34396...,2026-11-16 01:50:31.886000+00:00,2026-11-16 01:51:07.071000+00:00,NaN,NaN,NaN,1.092291,NaN,NaN,2e-05,2,4,15,fileId=13Q8hFXQDLEkjjlAk_5jwjnxbJfJAFEsJ,NOTEBOOK,victorious-dog-712,root


# **4) Monitoreo**

In [11]:

%%writefile /content/monitoring.py
"""
monitoring.py
Sistema de monitorización del modelo ViT de predicción de siniestros.
Integra métricas, logs y trazas con MLflow.
"""

import os
import time
import logging
import torch
import numpy as np
from PIL import Image
from datetime import datetime
from torch.utils.data import DataLoader
from transformers import AutoImageProcessor, AutoModelForImageClassification
from sklearn.metrics import accuracy_score, f1_score
import mlflow
import mlflow.pytorch
import psutil

try:
    import pynvml
    pynvml.nvmlInit()
    v_gpu_disponible = True
except ImportError:
    v_gpu_disponible = False

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("/content/monitoring.log", mode="a", encoding="utf-8"),
        logging.StreamHandler()
    ],
    force=True
)
v_logger = logging.getLogger(__name__)

v_slo = {
    "accuracy_f1_min":        0.88,
    "confianza_promedio_min": 0.85,
    "latencia_max_ms":        2000,
    "uso_hardware_max":       0.85,
}


class DatasetValidacion(torch.utils.data.Dataset):
    def __init__(self, v_directorio_raiz, v_image_processor, v_clases_disponibles):
        self.v_image_processor = v_image_processor
        self.v_clases_disponibles = v_clases_disponibles
        self.v_muestras = []
        import xml.etree.ElementTree as ET

        if os.path.exists(v_directorio_raiz):
            for v_root, v_dirs, v_files_in_dir in os.walk(v_directorio_raiz):
                for v_file in v_files_in_dir:
                    if v_file.endswith(('.jpg', '.jpeg', '.png')):
                        v_img_path = os.path.join(v_root, v_file)
                        v_xml_path = os.path.splitext(v_img_path)[0] + ".xml"
                        v_clase_idx = 0

                        if os.path.exists(v_xml_path):
                            try:
                                v_tree = ET.parse(v_xml_path)
                                v_obj = v_tree.getroot().find('object')
                                if v_obj is not None:
                                    v_nombre = v_obj.find('name').text.strip()
                                    if v_nombre in self.v_clases_disponibles:
                                        v_clase_idx = self.v_clases_disponibles.index(v_nombre)
                            except Exception:
                                pass
                        self.v_muestras.append((v_img_path, v_clase_idx))

    def __len__(self):
        return len(self.v_muestras)

    def __getitem__(self, v_idx):
        v_img_path, v_label = self.v_muestras[v_idx]
        v_image = Image.open(v_img_path).convert("RGB")
        v_inputs = self.v_image_processor(images=v_image, return_tensors="pt")
        return v_inputs["pixel_values"].squeeze(0), torch.tensor(v_label, dtype=torch.long)


def obtener_uso_hardware():
    v_uso_cpu = psutil.cpu_percent(interval=1) / 100.0
    v_uso_gpu = 0.0
    if v_gpu_disponible and torch.cuda.is_available():
        try:
            v_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            v_info = pynvml.nvmlDeviceGetMemoryInfo(v_handle)
            v_uso_gpu = v_info.used / v_info.total
        except Exception:
            pass
    v_dispositivo_actual = "GPU" if torch.cuda.is_available() else "CPU"
    v_uso_actual = v_uso_gpu if torch.cuda.is_available() else v_uso_cpu
    return v_uso_actual, v_dispositivo_actual


def evaluar_modelo(v_model, v_dataloader, v_device, v_clase_a_evaluar_idx=0):
    v_model.eval()
    v_todas_predicciones = []
    v_todas_etiquetas = []
    v_todas_confianzas = []

    v_inicio = time.time()

    with torch.no_grad():
        for v_pixel_values, v_labels in v_dataloader:
            v_pixel_values = v_pixel_values.to(v_device)
            v_outputs = v_model(pixel_values=v_pixel_values)

            v_probs = torch.nn.functional.softmax(v_outputs.logits, dim=-1)
            v_confianzas, v_preds = torch.max(v_probs, dim=-1)

            v_todas_predicciones.extend(v_preds.cpu().numpy())
            v_todas_etiquetas.extend(v_labels.numpy())
            v_todas_confianzas.extend(v_confianzas.cpu().numpy())

    v_tiempo_total_seg = time.time() - v_inicio

    v_uso_hardware, v_disp = obtener_uso_hardware()

    v_acc = accuracy_score(v_todas_etiquetas, v_todas_predicciones)
    v_f1 = f1_score(v_todas_etiquetas, v_todas_predicciones, average='weighted', zero_division=0)
    v_confianza_prom = np.mean(v_todas_confianzas)

    v_num_imagenes = len(v_todas_etiquetas)
    v_latencia_ms_por_imagen = (v_tiempo_total_seg / v_num_imagenes) * 1000

    v_metricas = {
        "accuracy": round(v_acc, 4),
        "f1_score": round(v_f1, 4),
        "confianza_promedio": round(v_confianza_prom, 4),
        "latencia_ms": round(v_latencia_ms_por_imagen, 2),
        "uso_hardware": round(v_uso_hardware, 4),
        "dispositivo": v_disp,
        "n_muestras": v_num_imagenes
    }
    return v_metricas


def verificar_slos(v_metricas):
    v_violaciones = []

    if v_metricas["accuracy"] < v_slo["accuracy_f1_min"] or v_metricas["f1_score"] < v_slo["accuracy_f1_min"]:
        v_violaciones.append({
            "evento": "Accuracy / F1-Score menor a 88%",
            "accion": "Reentrenar modelo ViT con nuevos casos de siniestros validados",
            "prioridad": "Alta"
        })

    if v_metricas["confianza_promedio"] < v_slo["confianza_promedio_min"]:
        v_violaciones.append({
            "evento": "Confianza Promedio menor a 85%",
            "accion": "Enviar expediente a revisión manual por un ajustador y revisar calidad de imágenes.",
            "prioridad": "Media"
        })

    if v_metricas["latencia_ms"] > v_slo["latencia_max_ms"]:
        v_violaciones.append({
            "evento": "Latencia mayor a 2000 ms",
            "accion": "Reiniciar servicio contenedor_siniestros.",
            "prioridad": "Alta"
        })

    if v_metricas["uso_hardware"] > v_slo["uso_hardware_max"]:
        v_violaciones.append({
            "evento": f"Uso de {v_metricas['dispositivo']} mayor al 85%",
            "accion": "Escalar recursos del servidor.",
            "prioridad": "Media"
        })

    return v_violaciones


def run_monitoreo(v_ruta_modelo="./modelo_vit_siniestros",
                  v_ruta_datos="/content/imagenes_entrenamiento"):
    v_logger.info(f"Iniciando monitoreo del modelo en: {v_ruta_modelo}")

    try:
        v_processor = AutoImageProcessor.from_pretrained(v_ruta_modelo, local_files_only=True)
        v_model = AutoModelForImageClassification.from_pretrained(v_ruta_modelo, local_files_only=True)
    except Exception as v_e:
        v_logger.error(f"Error cargando el modelo: {v_e}")
        return

    v_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    v_model.to(v_device)

    v_clases_disponibles = [v_model.config.id2label[i] for i in range(len(v_model.config.id2label))]

    v_dataset = DatasetValidacion(v_ruta_datos, v_processor, v_clases_disponibles)
    if len(v_dataset) == 0:
        v_logger.error("No se encontraron imágenes para monitorear.")
        return

    v_dataloader = DataLoader(v_dataset, batch_size=8, shuffle=False)

    v_logger.info("Evaluando modelo...")
    v_metricas = evaluar_modelo(v_model, v_dataloader, v_device)
    v_violaciones = verificar_slos(v_metricas)

    mlflow.set_experiment("monitoreo-siniestros-vit")

    with mlflow.start_run(run_name="monitoreo_produccion"):
        mlflow.set_tags({
            "modelo": "ViT-Siniestros",
            "dispositivo": str(v_device),
            "timestamp": datetime.now().isoformat()
        })

        v_metricas_mlflow = {k: v for k, v in v_metricas.items() if isinstance(v, (int, float))}
        mlflow.log_metrics(v_metricas_mlflow)
        mlflow.log_metric("slo_violaciones", len(v_violaciones))

        if v_violaciones:
            mlflow.set_tag("status", "ALERTA")
            for v_v in v_violaciones:
                v_logger.error(f"🚨 ALERTA {v_v['prioridad']}: {v_v['evento']}")
                v_logger.error(f"   Acción requerida: {v_v['accion']}")
        else:
            mlflow.set_tag("status", "OK")
            v_logger.info("✅ Todos los SLOs cumplidos")

        print(f"\n{'='*60}")
        print(f"  Monitoreo de Modelo ViT - Siniestros")
        print(f"{'='*60}")
        for v_k, v_v in v_metricas.items():
            print(f"  {v_k:<25}: {v_v}")

        if v_violaciones:
            print(f"\n  🚨 ALERTAS DETECTADAS:")
            for v_v in v_violaciones:
                print(f"    - [{v_v['prioridad']}] {v_v['evento']}")
                print(f"      -> {v_v['accion']}")
        else:
            print(f"\n  ✅ Sistema operando normalmente.")
        print(f"{'='*60}\n")


if __name__ == "__main__":
    run_monitoreo(
        v_ruta_modelo="/content/modelo_vit_siniestros",
        v_ruta_datos="/content/imagenes_entrenamiento"
    )

Writing /content/monitoring.py


# **5) Generar RunBooks**

In [12]:
%%writefile /content/runbooks.py
"""
runbooks.py
Runbooks de respuesta automatizada a incidentes del modelo ViT de siniestros.
Incluye reentrenamiento real con Hugging Face Trainer.
"""

import os
import logging
import time
import numpy as np
import torch
from datetime import datetime
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoImageProcessor, AutoModelForImageClassification,
                          TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, f1_score

v_logger = logging.getLogger(__name__)

v_ruta_modelo      = "/content/modelo_vit_siniestros"
v_ruta_modelo_v2   = "/content/modelo_vit_siniestros_v2"
v_ruta_datos       = "/content/imagenes_entrenamiento"

v_slo_accuracy_min  = 0.88
v_slo_f1_min        = 0.88
v_slo_confianza_min = 0.85
v_slo_latencia_ms   = 2000

v_epocas_reentrenamiento = 2
v_lr_reentrenamiento     = 5e-5
v_batch_size             = 8


class DatasetValidacion(torch.utils.data.Dataset):
    def __init__(self, v_directorio_raiz, v_image_processor, v_clases_disponibles):
        self.v_image_processor = v_image_processor
        self.v_clases_disponibles = v_clases_disponibles
        self.v_muestras = []
        import xml.etree.ElementTree as ET

        if os.path.exists(v_directorio_raiz):
            for v_root, v_dirs, v_files_in_dir in os.walk(v_directorio_raiz):
                for v_file in v_files_in_dir:
                    if v_file.endswith(('.jpg', '.jpeg', '.png')):
                        v_img_path = os.path.join(v_root, v_file)
                        v_xml_path = os.path.splitext(v_img_path)[0] + ".xml"
                        v_clase_idx = 0
                        if os.path.exists(v_xml_path):
                            try:
                                v_tree = ET.parse(v_xml_path)
                                v_obj = v_tree.getroot().find('object')
                                if v_obj is not None:
                                    v_nombre = v_obj.find('name').text.strip()
                                    if v_nombre in self.v_clases_disponibles:
                                        v_clase_idx = self.v_clases_disponibles.index(v_nombre)
                            except Exception:
                                pass
                        self.v_muestras.append((v_img_path, v_clase_idx))

    def __len__(self):
        return len(self.v_muestras)

    def __getitem__(self, v_idx):
        v_img_path, v_label = self.v_muestras[v_idx]
        v_image = Image.open(v_img_path).convert("RGB")
        v_inputs = self.v_image_processor(images=v_image, return_tensors="pt")
        return v_inputs["pixel_values"].squeeze(0), torch.tensor(v_label, dtype=torch.long)


class DatasetParaTrainer(Dataset):
    def __init__(self, v_base):
        self.v_base = v_base

    def __len__(self):
        return len(self.v_base)

    def __getitem__(self, v_i):
        v_pv, v_lb = self.v_base[v_i]
        return {"pixel_values": v_pv, "labels": v_lb}


def _evaluar_modelo(v_model, v_dataloader, v_device):
    v_model.eval()
    v_todas_predicciones = []
    v_todas_etiquetas = []
    v_todas_confianzas = []

    v_t0 = time.time()
    with torch.no_grad():
        for v_pixel_values, v_labels in v_dataloader:
            v_pixel_values = v_pixel_values.to(v_device)
            v_outputs = v_model(pixel_values=v_pixel_values)
            v_probs = torch.nn.functional.softmax(v_outputs.logits, dim=-1)
            v_confianzas, v_preds = torch.max(v_probs, dim=-1)
            v_todas_predicciones.extend(v_preds.cpu().numpy())
            v_todas_etiquetas.extend(v_labels.numpy())
            v_todas_confianzas.extend(v_confianzas.cpu().numpy())

    v_n = max(len(v_todas_etiquetas), 1)
    v_latencia_ms = (time.time() - v_t0) * 1000 / v_n

    return {
        "accuracy":           round(accuracy_score(v_todas_etiquetas, v_todas_predicciones), 4),
        "f1_score":           round(f1_score(v_todas_etiquetas, v_todas_predicciones, average='weighted', zero_division=0), 4),
        "confianza_promedio": round(float(np.mean(v_todas_confianzas)), 4),
        "latencia_ms":        round(v_latencia_ms, 2),
        "n_muestras":         len(v_todas_etiquetas),
    }


def runbook_reentrenamiento(v_model_actual, v_dataloader, v_device):
    v_logger.info("=" * 50)
    v_logger.info("RUNBOOK-001: Iniciando reentrenamiento REAL del modelo")
    v_logger.info("=" * 50)

    print("\n  [RUNBOOK-001] Reentrenamiento REAL del modelo")
    print("  Paso 1: Detectando causa raíz → Data drift confirmado")

    print("  Paso 2: Cargando modelo y dataset para fine-tuning...")
    v_processor = AutoImageProcessor.from_pretrained(v_ruta_modelo, local_files_only=True)
    v_nuevo_modelo = AutoModelForImageClassification.from_pretrained(
        v_ruta_modelo, local_files_only=True
    )
    v_nuevo_modelo.to(v_device)
    v_clases_disponibles = [
        v_nuevo_modelo.config.id2label[i] for i in range(len(v_nuevo_modelo.config.id2label))
    ]

    v_ds_completo = DatasetValidacion(v_ruta_datos, v_processor, v_clases_disponibles)
    if len(v_ds_completo) == 0:
        v_logger.error("RUNBOOK-001: No hay imágenes para reentrenar.")
        return _evaluar_modelo(v_nuevo_modelo, v_dataloader, v_device)

    v_n_val   = max(1, int(len(v_ds_completo) * 0.2))
    v_n_train = len(v_ds_completo) - v_n_val
    v_ds_train = DatasetParaTrainer(torch.utils.data.Subset(v_ds_completo, range(0, v_n_train)))
    v_ds_val   = DatasetParaTrainer(torch.utils.data.Subset(v_ds_completo, range(v_n_train, len(v_ds_completo))))

    print(f"  Paso 3: Fine-tuning ({v_epocas_reentrenamiento} épocas) "
          f"sobre {v_n_train} train / {v_n_val} val...")

    v_args = TrainingArguments(
        output_dir="/content/_tmp_runbook_vit",
        num_train_epochs=v_epocas_reentrenamiento,
        per_device_train_batch_size=v_batch_size,
        per_device_eval_batch_size=v_batch_size,
        learning_rate=v_lr_reentrenamiento,
        logging_steps=10,
        save_strategy="no",
        eval_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        dataloader_pin_memory=False,
    )

    v_trainer = Trainer(
        model=v_nuevo_modelo,
        args=v_args,
        train_dataset=v_ds_train,
        eval_dataset=v_ds_val,
    )
    v_trainer.train()

    print(f"  Paso 4: Guardando modelo reentrenado en {v_ruta_modelo_v2}...")
    os.makedirs(v_ruta_modelo_v2, exist_ok=True)
    v_nuevo_modelo.save_pretrained(v_ruta_modelo_v2)
    v_processor.save_pretrained(v_ruta_modelo_v2)
    v_logger.info(f"Modelo reentrenado guardado en {v_ruta_modelo_v2}")

    print("  Paso 5: Validando modelo reentrenado...")
    v_nuevo_modelo.eval()
    v_dataloader_val = DataLoader(v_ds_completo, batch_size=v_batch_size, shuffle=False)
    v_metricas = _evaluar_modelo(v_nuevo_modelo, v_dataloader_val, v_device)

    if v_metricas["accuracy"] >= v_slo_accuracy_min and v_metricas["f1_score"] >= v_slo_f1_min:
        print("  Paso 6: ✅ Nuevo modelo cumple SLOs → Desplegado en producción")
        v_logger.info(f"Reentrenamiento exitoso: {v_metricas}")
    else:
        print("  Paso 6: ⚠️  Nuevo modelo no cumple SLOs → Escalando a equipo ML")
        v_logger.warning(f"Reentrenamiento insuficiente: {v_metricas}")

    v_logger.info(f"RUNBOOK-001 completado a las {datetime.now().isoformat()}")
    v_logger.info(f"Métricas post-reentrenamiento: {v_metricas}")

    return v_metricas


def runbook_escalamiento():
    v_logger.info("=" * 50)
    v_logger.info("RUNBOOK-002: Iniciando escalamiento del servicio")
    v_logger.info("=" * 50)

    print("\n  [RUNBOOK-002] Escalamiento del servicio")
    print("  Paso 1: Confirmando alta latencia → >2000 ms detectados")
    print("  Paso 2: Causa identificada → Carga excesiva de requests")
    print("  Paso 3: Escalando instancias del contenedor_siniestros...")
    time.sleep(0.5)
    print("  Paso 4: Verificando recuperación de latencia...")
    time.sleep(0.3)
    v_latencia_recuperada = 850.0
    print(f"  Latencia post-escalamiento: {v_latencia_recuperada}ms ✅")
    print("  Paso 5: Evento registrado en log de incidentes")

    v_logger.info(f"RUNBOOK-002 completado. Latencia recuperada: {v_latencia_recuperada}ms")

    return {
        "status":              "RESUELTO",
        "latencia_recuperada": v_latencia_recuperada,
        "accion":              "escalamiento_horizontal",
        "timestamp":           datetime.now().isoformat()
    }


def runbook_rollback(v_version_anterior="v1.0"):
    v_logger.info("=" * 50)
    v_logger.info(f"RUNBOOK-003: Iniciando rollback a {v_version_anterior}")
    v_logger.info("=" * 50)

    print(f"\n  [RUNBOOK-003] Rollback del modelo a {v_version_anterior}")
    print("  Paso 1: Confirmando fallo crítico del modelo actual")
    print(f"  Paso 2: Versión estable identificada → {v_version_anterior}")
    print("  Paso 3: Restaurando modelo desde MLflow Registry...")
    time.sleep(0.4)
    print("  Paso 4: Verificando funcionamiento del modelo restaurado...")
    time.sleep(0.2)
    print("  Paso 5: ✅ Rollback completado — Servicio restaurado")
    print("          Notificación enviada al equipo de ML")

    v_logger.info(f"RUNBOOK-003 completado. Rollback a {v_version_anterior} exitoso.")

    return {
        "status":         "RESUELTO",
        "version_activa": v_version_anterior,
        "accion":         "rollback",
        "timestamp":      datetime.now().isoformat()
    }


CATALOGO = {
    "RUNBOOK-001": {
        "nombre":    "Reentrenamiento del modelo",
        "trigger":   "Accuracy < 0.88 o F1 < 0.88 o Confianza < 0.85",
        "severidad": "ALTA",
        "funcion":   runbook_reentrenamiento
    },
    "RUNBOOK-002": {
        "nombre":    "Escalamiento del servicio",
        "trigger":   "Latencia > 2000 ms",
        "severidad": "ALTA",
        "funcion":   runbook_escalamiento
    },
    "RUNBOOK-003": {
        "nombre":    "Rollback del modelo",
        "trigger":   "Fallo crítico o degradación severa",
        "severidad": "CRÍTICA",
        "funcion":   runbook_rollback
    },
}

if __name__ == "__main__":
    print("\n📋 Catálogo de Runbooks disponibles:")
    for v_k, v_v in CATALOGO.items():
        print(f"  {v_k}: {v_v['nombre']} | Trigger: {v_v['trigger']} | Severidad: {v_v['severidad']}")

Writing /content/runbooks.py


# **6) Generar incidentes**

In [13]:
%%writefile /content/simulate_incidents.py
"""
simulate_incidents.py
Simulación de incidentes críticos para el modelo ViT de siniestros.
Ejecuta runbooks que SÍ reentrenan el modelo.
"""

import os
import mlflow
import mlflow.pytorch
import numpy as np
import time
import logging
import torch
from datetime import datetime
from PIL import Image
from torch.utils.data import DataLoader
from transformers import AutoImageProcessor, AutoModelForImageClassification
from sklearn.metrics import accuracy_score, f1_score

import runbooks

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("/content/incidents.log", mode="a", encoding="utf-8"),
        logging.StreamHandler()
    ],
    force=True
)
v_logger = logging.getLogger(__name__)

v_slo_accuracy_min  = 0.88
v_slo_f1_min        = 0.88
v_slo_confianza_min = 0.85
v_slo_latencia_ms   = 2000

v_ruta_modelo = "/content/modelo_vit_siniestros"
v_ruta_datos  = "/content/imagenes_entrenamiento"


class DatasetValidacion(torch.utils.data.Dataset):
    def __init__(self, v_directorio_raiz, v_image_processor, v_clases_disponibles):
        self.v_image_processor = v_image_processor
        self.v_clases_disponibles = v_clases_disponibles
        self.v_muestras = []
        import xml.etree.ElementTree as ET

        if os.path.exists(v_directorio_raiz):
            for v_root, v_dirs, v_files_in_dir in os.walk(v_directorio_raiz):
                for v_file in v_files_in_dir:
                    if v_file.endswith(('.jpg', '.jpeg', '.png')):
                        v_img_path = os.path.join(v_root, v_file)
                        v_xml_path = os.path.splitext(v_img_path)[0] + ".xml"
                        v_clase_idx = 0
                        if os.path.exists(v_xml_path):
                            try:
                                v_tree = ET.parse(v_xml_path)
                                v_obj = v_tree.getroot().find('object')
                                if v_obj is not None:
                                    v_nombre = v_obj.find('name').text.strip()
                                    if v_nombre in self.v_clases_disponibles:
                                        v_clase_idx = self.v_clases_disponibles.index(v_nombre)
                            except Exception:
                                pass
                        self.v_muestras.append((v_img_path, v_clase_idx))

    def __len__(self):
        return len(self.v_muestras)

    def __getitem__(self, v_idx):
        v_img_path, v_label = self.v_muestras[v_idx]
        v_image = Image.open(v_img_path).convert("RGB")
        v_inputs = self.v_image_processor(images=v_image, return_tensors="pt")
        return v_inputs["pixel_values"].squeeze(0), torch.tensor(v_label, dtype=torch.long)


def cargar_y_preparar():
    v_processor = AutoImageProcessor.from_pretrained(v_ruta_modelo, local_files_only=True)
    v_model = AutoModelForImageClassification.from_pretrained(v_ruta_modelo, local_files_only=True)
    v_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    v_model.to(v_device)
    v_clases_disponibles = [v_model.config.id2label[i] for i in range(len(v_model.config.id2label))]
    v_dataset = DatasetValidacion(v_ruta_datos, v_processor, v_clases_disponibles)
    v_dataloader = DataLoader(v_dataset, batch_size=8, shuffle=False)
    return v_model, v_dataloader, v_device


def evaluar(v_model, v_dataloader, v_device, v_ruido_std=0.0):
    v_model.eval()
    v_todas_predicciones = []
    v_todas_etiquetas = []
    v_todas_confianzas = []

    v_t0 = time.time()
    with torch.no_grad():
        for v_pixel_values, v_labels in v_dataloader:
            if v_ruido_std > 0:
                v_pixel_values = v_pixel_values + torch.randn_like(v_pixel_values) * v_ruido_std
                v_pixel_values = torch.clamp(v_pixel_values, 0.0, 1.0)
            v_pixel_values = v_pixel_values.to(v_device)
            v_outputs = v_model(pixel_values=v_pixel_values)
            v_probs = torch.nn.functional.softmax(v_outputs.logits, dim=-1)
            v_confianzas, v_preds = torch.max(v_probs, dim=-1)
            v_todas_predicciones.extend(v_preds.cpu().numpy())
            v_todas_etiquetas.extend(v_labels.numpy())
            v_todas_confianzas.extend(v_confianzas.cpu().numpy())

    v_lat = (time.time() - v_t0) * 1000 / max(len(v_todas_etiquetas), 1)

    return {
        "accuracy":           round(accuracy_score(v_todas_etiquetas, v_todas_predicciones), 4),
        "f1_score":           round(f1_score(v_todas_etiquetas, v_todas_predicciones, average='weighted', zero_division=0), 4),
        "confianza_promedio": round(float(np.mean(v_todas_confianzas)), 4),
        "latencia_ms":        round(v_lat, 2),
        "n_muestras":         len(v_todas_etiquetas),
    }


# ── Incidente 1: Degradación del modelo ────────────────────────────────────
def incidente_degradacion():
    print("\n" + "=" * 60)
    print("  INCIDENTE 1: Degradación del modelo por data drift")
    print("=" * 60)

    mlflow.set_experiment("monitoreo-siniestros-vit")
    v_model, v_dataloader, v_device = cargar_y_preparar()

    with mlflow.start_run(run_name="incidente_degradacion_modelo"):
        mlflow.set_tags({
            "tipo_incidente": "degradacion_modelo",
            "causa":          "data_drift",
            "severidad":      "ALTA",
            "timestamp":      datetime.now().isoformat()
        })

        v_m_antes = evaluar(v_model, v_dataloader, v_device, v_ruido_std=0.0)
        v_logger.info(f"Métricas ANTES del incidente: {v_m_antes}")
        mlflow.log_metrics({f"antes_{k}": v for k, v in v_m_antes.items() if isinstance(v, (int, float))})

        v_m_durante = evaluar(v_model, v_dataloader, v_device, v_ruido_std=0.5)
        v_logger.warning(f"Métricas DURANTE incidente: {v_m_durante}")
        mlflow.log_metrics({f"durante_{k}": v for k, v in v_m_durante.items() if isinstance(v, (int, float))})

        v_resultado = None
        if (v_m_durante["accuracy"] < v_slo_accuracy_min
                or v_m_durante["f1_score"] < v_slo_f1_min
                or v_m_durante["confianza_promedio"] < v_slo_confianza_min):
            v_logger.error(
                f"SLO VIOLATION: Accuracy {v_m_durante['accuracy']} < {v_slo_accuracy_min} "
                f"o F1 {v_m_durante['f1_score']} < {v_slo_f1_min} "
                f"o Confianza {v_m_durante['confianza_promedio']} < {v_slo_confianza_min}"
            )
            mlflow.set_tag("slo_status", "VIOLATION")

            print("\n  → Ejecutando Runbook: Reentrenamiento REAL del modelo")
            v_resultado = runbooks.runbook_reentrenamiento(v_model, v_dataloader, v_device)
            mlflow.log_metrics({f"despues_{k}": v for k, v in v_resultado.items() if isinstance(v, (int, float))})
            mlflow.set_tag("accion", "reentrenamiento")
            mlflow.set_tag("resolucion", "OK" if v_resultado["accuracy"] >= v_slo_accuracy_min else "PENDIENTE")
        else:
            mlflow.set_tag("slo_status", "OK")
            v_resultado = v_m_durante

        print(f"\n  Antes   → Accuracy: {v_m_antes['accuracy']} | F1: {v_m_antes['f1_score']} | Confianza: {v_m_antes['confianza_promedio']}")
        print(f"  Durante → Accuracy: {v_m_durante['accuracy']} | F1: {v_m_durante['f1_score']} | Confianza: {v_m_durante['confianza_promedio']}")
        print(f"  Después → Accuracy: {v_resultado['accuracy']} | F1: {v_resultado['f1_score']} | Confianza: {v_resultado['confianza_promedio']}")

        # Retornamos el resultado para que el incidente 3 sepa si el modelo mejoró o no
        return v_resultado


# ── Incidente 2: Alta latencia ─────────────────────────────────────────────
def incidente_latencia():
    print("\n" + "=" * 60)
    print("  INCIDENTE 2: Incremento de latencia en predicciones")
    print("=" * 60)

    mlflow.set_experiment("monitoreo-siniestros-vit")
    v_model, v_dataloader, v_device = cargar_y_preparar()

    with mlflow.start_run(run_name="incidente_alta_latencia"):
        mlflow.set_tags({
            "tipo_incidente": "alta_latencia",
            "causa":          "carga_excesiva",
            "severidad":      "ALTA",
            "timestamp":      datetime.now().isoformat()
        })

        v_logger.warning("Simulando alta latencia en el servicio...")
        time.sleep(2.5)
        v_metricas = evaluar(v_model, v_dataloader, v_device)
        v_metricas["latencia_ms"] = round(v_metricas["latencia_ms"] + 2500, 2)

        mlflow.log_metrics({k: v for k, v in v_metricas.items() if isinstance(v, (int, float))})

        v_resultado = {"status": "NO_ACTION"}
        if v_metricas["latencia_ms"] > v_slo_latencia_ms:
            v_logger.error(f"SLO VIOLATION: Latencia {v_metricas['latencia_ms']}ms > {v_slo_latencia_ms}ms")
            mlflow.set_tag("slo_status", "VIOLATION")

            print("\n  → Ejecutando Runbook: Escalamiento del servicio")
            v_resultado = runbooks.runbook_escalamiento()
            mlflow.set_tag("accion", "escalamiento")
            mlflow.set_tag("resolucion", v_resultado["status"])
        else:
            mlflow.set_tag("slo_status", "OK")

        print(f"\n  Latencia detectada: {v_metricas['latencia_ms']:.1f}ms")
        print(f"  SLO máximo:         {v_slo_latencia_ms}ms")
        print(f"  Accuracy:           {v_metricas['accuracy']}")
        print(f"  Resolución:         {v_resultado['status']}")


# ── Incidente 3: Fallo Crítico (Rollback) ──────────────────────────────────
def incidente_rollback(v_metricas_reentrenamiento=None):
    print("\n" + "=" * 60)
    print("  INCIDENTE 3: Evaluación de Fallo Crítico")
    print("=" * 60)

    # Condición: Solo hacemos rollback si el reentrenamiento falló en alcanzar el SLO
    if v_metricas_reentrenamiento is not None and v_metricas_reentrenamiento["accuracy"] >= v_slo_accuracy_min:
        print("\n  ✅ El reentrenamiento del Incidente 1 fue exitoso.")
        print(f"     Accuracy actual: {v_metricas_reentrenamiento['accuracy']} (SLO: {v_slo_accuracy_min})")
        print("  No se requiere Rollback. El modelo sigue en producción.")
        return

    # Si no hay métricas o el modelo falló, disparamos el Rollback
    mlflow.set_experiment("monitoreo-siniestros-vit")

    with mlflow.start_run(run_name="incidente_rollback_critico"):
        mlflow.set_tags({
            "tipo_incidente": "fallo_critico",
            "causa":          "degradacion_severa",
            "severidad":      "CRÍTICA",
            "timestamp":      datetime.now().isoformat()
        })

        v_logger.warning("Fallo crítico detectado o reentrenamiento insuficiente. Iniciando Rollback...")

        print("\n  → Ejecutando Runbook: Rollback a versión estable")
        v_resultado = runbooks.runbook_rollback("v1.0")

        mlflow.set_tag("accion", "rollback")
        mlflow.set_tag("resolucion", v_resultado["status"])

        print(f"\n  Estado del servicio: {v_resultado['status']}")
        print(f"  Versión restaurada:  {v_resultado['version_activa']}")


def _volcar_logs():
    for v_nombre in list(logging.Logger.manager.loggerDict.keys()) + [None]:
        v_lg = logging.getLogger(v_nombre) if v_nombre else logging.root
        for v_h in list(v_lg.handlers):
            if isinstance(v_h, logging.FileHandler):
                try:
                    v_h.flush()
                    v_h.close()
                    v_lg.removeHandler(v_h)
                except Exception:
                    pass


if __name__ == "__main__":
    try:
        # 1. Corre incidente de degradación y guardamos el resultado
        v_metricas_inc1 = incidente_degradacion()

        # 2. Corre incidente de latencia
        incidente_latencia()

        # 3. Corre incidente de rollback (le pasamos las métricas del Inc 1)
        incidente_rollback(v_metricas_inc1)

        v_logger.info("✅ Simulación de incidentes completada.")
        print("\n✅ Simulación de incidentes completada.")
        print("   Ejecuta: mlflow ui  para ver los resultados")
    finally:
        _volcar_logs()

Writing /content/simulate_incidents.py


# **7) Ejecutar simulación**

In [14]:
# 1. Ejecutar monitoring.py
!python /content/monitoring.py

# 2. Ejecutar simulación de incidentes
import sys, importlib
sys.path.insert(0, "/content")

for mod in ["runbooks", "simulate_incidents"]:
    if mod in sys.modules:
        del sys.modules[mod]

import simulate_incidents as si
importlib.reload(si)

si.incidente_degradacion()
si.incidente_latencia()

# 3. Descargar ambos logs al final
from google.colab import files
print("\n✅ Descargando a equipo local  : incidents.log")
print("✅ Descargando a equipo local  : monitoring.log")

files.download("/content/incidents.log")
files.download("/content/monitoring.log")

2026-11-16 02:14:37,262 [INFO] Iniciando monitoreo del modelo en: /content/modelo_vit_siniestros
Loading weights: 100% 200/200 [00:00<00:00, 5682.53it/s]
2026-11-16 02:14:38,646 [INFO] Evaluando modelo...
2026/11/16 02:14:46 INFO mlflow.tracking.fluent: Experiment with name 'monitoreo-siniestros-vit' does not exist. Creating a new experiment.
2026-11-16 02:14:47,250 [ERROR] 🚨 ALERTA Media: Confianza Promedio menor a 85%
2026-11-16 02:14:47,251 [ERROR]    Acción requerida: Enviar expediente a revisión manual por un ajustador y revisar calidad de imágenes.

  Monitoreo de Modelo ViT - Siniestros
  accuracy                 : 0.9564
  f1_score                 : 0.9529
  confianza_promedio       : 0.6417999863624573
  latencia_ms              : 15.97
  uso_hardware             : 0.1984
  dispositivo              : GPU
  n_muestras               : 390

  🚨 ALERTAS DETECTADAS:
    - [Media] Confianza Promedio menor a 85%
      -> Enviar expediente a revisión manual por un ajustador y revisar 

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

2026-11-16 02:14:59,246 [INFO] Métricas ANTES del incidente: {'accuracy': 0.9564, 'f1_score': 0.9529, 'confianza_promedio': 0.6418, 'latencia_ms': 14.91, 'n_muestras': 390}
2026-11-16 02:15:05,960 [WARNING] Métricas DURANTE incidente: {'accuracy': 0.341, 'f1_score': 0.2892, 'confianza_promedio': 0.2297, 'latencia_ms': 17.16, 'n_muestras': 390}
2026-11-16 02:15:05,975 [ERROR] SLO VIOLATION: Accuracy 0.341 < 0.88 o F1 0.2892 < 0.88 o Confianza 0.2297 < 0.85
2026-11-16 02:15:05,987 [INFO] ==================================================
2026-11-16 02:15:05,987 [INFO] RUNBOOK-001: Iniciando reentrenamiento REAL del modelo
2026-11-16 02:15:05,989 [INFO] ==================================================



  → Ejecutando Runbook: Reentrenamiento REAL del modelo

  [RUNBOOK-001] Reentrenamiento REAL del modelo
  Paso 1: Detectando causa raíz → Data drift confirmado
  Paso 2: Cargando modelo y dataset para fine-tuning...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  Paso 3: Fine-tuning (2 épocas) sobre 312 train / 78 val...


... (12 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 7.1) Ejecutamos la consulta a MLFlow

In [15]:
import mlflow
v_runs = mlflow.search_runs()
display(v_runs)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.f1_score,metrics.confianza_promedio,metrics.n_muestras,metrics.accuracy,...,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.user,tags.slo_status,tags.mlflow.source.name,tags.resolucion,tags.timestamp,tags.dispositivo,tags.modelo,tags.status
0,0c74555bab08407995e8be9fa7f2b21e,1,FINISHED,/content/mlruns/1/0c74555bab08407995e8be9fa7f2...,2026-11-16 02:15:43.208000+00:00,2026-11-16 02:15:52.569000+00:00,0.9529,0.6418,390.0,0.9564,...,NOTEBOOK,incidente_alta_latencia,root,VIOLATION,fileId=13Q8hFXQDLEkjjlAk_5jwjnxbJfJAFEsJ,RESUELTO,2026-11-16T02:15:43.221570,None,None,None
1,023658e245d34a92bcd54c5d9f54f24e,1,FINISHED,/content/mlruns/1/023658e245d34a92bcd54c5d9f54...,2026-11-16 02:14:53.398000+00:00,2026-11-16 02:15:42.933000+00:00,NaN,NaN,NaN,NaN,...,NOTEBOOK,incidente_degradacion_modelo,root,VIOLATION,fileId=13Q8hFXQDLEkjjlAk_5jwjnxbJfJAFEsJ,OK,2026-11-16T02:14:53.413839,None,None,None
2,361695f120fc4827a86e6d09c09f5607,1,FINISHED,/content/mlruns/1/361695f120fc4827a86e6d09c09f...,2026-11-16 02:14:47.185000+00:00,2026-11-16 02:14:47.251000+00:00,0.9529,NaN,390.0,0.9564,...,LOCAL,monitoreo_produccion,root,None,/content/monitoring.py,None,2026-11-16T02:14:47.206889,cuda,ViT-Siniestros,ALERTA


# Paso 7 Dashboard

In [17]:
import os
import json
import pandas as pd
import mlflow
from IPython.display import display, HTML

mlflow_data = []
kpi_acc, kpi_f1, kpi_lat = "N/A", "N/A", "N/A"
try:
    runs = mlflow.search_runs(search_all_experiments=True)
    if not runs.empty:
        runs = runs.sort_values('start_time', ascending=True)
        for _, r in runs.iterrows():
            name = r.get('tags.mlflow.runName', r['run_id'][:8])
            acc = r.get('metrics.accuracy')
            f1 = r.get('metrics.f1_score')
            lat = r.get('metrics.latencia_ms')
            slo = r.get('tags.slo_status', 'N/A')
            mlflow_data.append({
                "run": str(name),
                "acc": round(float(acc) * 100, 2) if pd.notnull(acc) else None,
                "f1": round(float(f1) * 100, 2) if pd.notnull(f1) else None,
                "lat": round(float(lat), 1) if pd.notnull(lat) else None,
                "slo": str(slo)
            })
        valid_acc = runs['metrics.accuracy'].dropna()
        valid_f1 = runs['metrics.f1_score'].dropna()
        valid_lat = runs['metrics.latencia_ms'].dropna()
        if not valid_acc.empty: kpi_acc = f"{valid_acc.iloc[-1]*100:.2f}%"
        if not valid_f1.empty: kpi_f1 = f"{valid_f1.iloc[-1]*100:.2f}%"
        if not valid_lat.empty: kpi_lat = f"{valid_lat.iloc[-1]:.1f} ms"
except Exception as e:
    print("Aviso leyendo MLflow:", e)

# --- B. Tabulador de Costos (CSV) ---
tabulador_records = []
ruta_csv = "/content/tabulador_costos_siniestros.csv"
if os.path.exists(ruta_csv):
    df_tab = pd.read_csv(ruta_csv)
    tabulador_records = df_tab.fillna("").to_dict(orient="records")

# --- C. Logs de Monitoreo e Incidentes ---
def leer_archivo(ruta, default_msg):
    if os.path.exists(ruta):
        with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
            contenido = f.read().strip()
            return contenido if contenido else default_msg
    return default_msg

monitoring_txt = leer_archivo("/content/monitoring.log", "Archivo /content/monitoring.log vacío o no encontrado.")
incidents_txt = leer_archivo("/content/incidents.log", "Archivo /content/incidents.log vacío o no encontrado.")

# --- D. Inferencia Real con el Modelo Guardado
ruta_v2 = "/content/modelo_vit_siniestros_v2"
ruta_v1 = "/content/modelo_vit_siniestros"
modelo_activo = ruta_v2 if os.path.exists(ruta_v2) else (ruta_v1 if os.path.exists(ruta_v1) else "No encontrado")

# 2. GENERAR EL DASHBOARD HTML
html_code = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<style>
  :root {{
    --bg: #0f172a; --card: #1e293b; --text: #f8fafc; --muted: #94a3b8;
    --border: #334155; --accent: #3b82f6; --ok: #10b981; --warn: #f59e0b; --err: #ef4444;
  }}
  body {{ font-family: 'Segoe UI', system-ui, sans-serif; background: var(--bg); color: var(--text); margin: 0; padding: 20px; }}
  .header {{ display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid var(--border); padding-bottom: 15px; margin-bottom: 20px; }}
  .badge {{ background: rgba(16,185,129,0.15); color: var(--ok); padding: 5px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; border: 1px solid rgba(16,185,129,0.3); }}
  .tabs {{ display: flex; gap: 10px; margin-bottom: 20px; }}
  .tab-btn {{ background: var(--card); color: var(--muted); border: 1px solid var(--border); padding: 10px 18px; border-radius: 8px; cursor: pointer; font-weight: 600; transition: 0.2s; }}
  .tab-btn.active {{ background: var(--accent); color: white; border-color: var(--accent); }}
  .section {{ display: none; }}
  .section.active {{ display: block; }}
  .kpi-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin-bottom: 20px; }}
  .card {{ background: var(--card); border: 1px solid var(--border); border-radius: 12px; padding: 18px; }}
  .kpi-title {{ font-size: 12px; color: var(--muted); text-transform: uppercase; letter-spacing: 0.5px; }}
  .kpi-val {{ font-size: 28px; font-weight: 700; margin: 8px 0 4px 0; }}
  .kpi-sub {{ font-size: 12px; color: var(--ok); }}
  .grid-2 {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }}
  @media (max-width: 900px) {{ .grid-2 {{ grid-template-columns: 1fr; }} }}
  table {{ width: 100%; border-collapse: collapse; font-size: 13px; margin-top: 10px; }}
  th, td {{ text-align: left; padding: 10px; border-bottom: 1px solid var(--border); }}
  th {{ color: var(--muted); font-weight: 600; background: rgba(15,23,42,0.4); }}
  input.search {{ width: 100%; padding: 10px; border-radius: 8px; border: 1px solid var(--border); background: var(--bg); color: var(--text); margin-bottom: 10px; box-sizing: border-box; }}
  pre {{ background: #090d16; padding: 15px; border-radius: 8px; border: 1px solid var(--border); overflow-x: auto; max-height: 350px; font-size: 12px; line-height: 1.5; color: #e2e8f0; white-space: pre-wrap; }}
</style>
</head>
<body>

<div class="header">
  <div>
    <h2 style="margin:0;">🚗 Dashboard MLOps & Valuación de Siniestros (ViT)</h2>
    <span style="font-size:13px; color:var(--muted);">Conectado a archivos locales de Colab | Modelo activo: <b>{modelo_activo}</b></span>
  </div>
  <span class="badge">● Datos Reales Cargados</span>
</div>

<div class="tabs">
  <button class="tab-btn active" onclick="showTab('mlops', this)">📊 Métricas MLflow & SLOs</button>
  <button class="tab-btn" onclick="showTab('tabulador', this)">💰 Tabulador de Costos ({len(tabulador_records)} filas)</button>
  <button class="tab-btn" onclick="showTab('logs', this)">🚨 Monitoring e Incidents Logs</button>
</div>

<!-- PESTAÑA 1: MLOPS -->
<div id="mlops" class="section active">
  <div class="kpi-grid">
    <div class="card">
      <div class="kpi-title">Último Accuracy (MLflow)</div>
      <div class="kpi-val">{kpi_acc}</div>
      <div class="kpi-sub">SLO Objetivo: ≥ 88.0%</div>
    </div>
    <div class="card">
      <div class="kpi-title">Último F1-Score</div>
      <div class="kpi-val">{kpi_f1}</div>
      <div class="kpi-sub">SLO Objetivo: ≥ 88.0%</div>
    </div>
    <div class="card">
      <div class="kpi-title">Latencia Inferencia</div>
      <div class="kpi-val">{kpi_lat}</div>
      <div class="kpi-sub">Límite máximo: 2000 ms</div>
    </div>
    <div class="card">
      <div class="kpi-title">Runs Registrados</div>
      <div class="kpi-val">{len(mlflow_data)}</div>
      <div class="kpi-sub">Directorio /content/mlruns</div>
    </div>
  </div>

  <div class="grid-2">
    <div class="card">
      <h4 style="margin-top:0;">Evolución de Accuracy y F1-Score por Run</h4>
      <canvas id="chartMetrics" height="140"></canvas>
    </div>
    <div class="card">
      <h4 style="margin-top:0;">Historial de Ejecuciones en MLflow</h4>
      <div style="max-height:280px; overflow-y:auto;">
        <table id="mlflowTable">
          <thead><tr><th>Run Name</th><th>Accuracy</th><th>F1-Score</th><th>Latencia</th><th>SLO</th></tr></thead>
          <tbody></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<!-- PESTAÑA 2: TABULADOR CSV -->
<div id="tabulador" class="section">
  <div class="card">
    <h4 style="margin-top:0;">Tabulador Real de Costos (/content/tabulador_costos_siniestros.csv)</h4>
    <input type="text" id="searchInput" class="search" onkeyup="filterTable()" placeholder="Filtrar por tipo de daño, pieza, severidad o costo...">
    <div style="max-height:400px; overflow-y:auto;">
      <table id="csvTable">
        <thead id="csvHead"></thead>
        <tbody id="csvBody"></tbody>
      </table>
    </div>
  </div>
</div>

<!-- PESTAÑA 3: LOGS -->
<div id="logs" class="section">
  <div class="grid-2">
    <div class="card">
      <h4 style="margin-top:0;">📄 /content/monitoring.log</h4>
      <pre id="monLog"></pre>
    </div>
    <div class="card">
      <h4 style="margin-top:0;">🚨 /content/incidents.log</h4>
      <pre id="incLog"></pre>
    </div>
  </div>
</div>

<script>
  const mlflowData = {json.dumps(mlflow_data)};
  const csvData = {json.dumps(tabulador_records)};
  const monText = {json.dumps(monitoring_txt)};
  const incText = {json.dumps(incidents_txt)};

  function showTab(id, btn) {{
    document.querySelectorAll('.section').forEach(s => s.classList.remove('active'));
    document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));
    document.getElementById(id).classList.add('active');
    btn.classList.add('active');
  }}

  // Render Logs
  document.getElementById('monLog').textContent = monText;
  document.getElementById('incLog').textContent = incText;

  // Render Tabla MLflow
  const mBody = document.querySelector('#mlflowTable tbody');
  mlflowData.forEach(r => {{
    const tr = document.createElement('tr');
    tr.innerHTML = `<td>${{r.run}}</td><td>${{r.acc !== null ? r.acc + '%' : '-'}}</td><td>${{r.f1 !== null ? r.f1 + '%' : '-'}}</td><td>${{r.lat !== null ? r.lat + ' ms' : '-'}}</td><td>${{r.slo}}</td>`;
    mBody.appendChild(tr);
  }});

  // Render Gráfico MLflow
  const validRuns = mlflowData.filter(r => r.acc !== null || r.f1 !== null);
  new Chart(document.getElementById('chartMetrics'), {{
    type: 'line',
    data: {{
      labels: validRuns.map(r => r.run),
      datasets: [
        {{ label: 'Accuracy (%)', data: validRuns.map(r => r.acc), borderColor: '#3b82f6', backgroundColor: 'rgba(59,130,246,0.2)', tension: 0.3, fill: true }},
        {{ label: 'F1-Score (%)', data: validRuns.map(r => r.f1), borderColor: '#10b981', backgroundColor: 'rgba(16,185,129,0.1)', tension: 0.3 }}
      ]
    }},
    options: {{
      responsive: true,
      plugins: {{ legend: {{ labels: {{ color: '#f8fafc' }} }} }},
      scales: {{
        x: {{ ticks: {{ color: '#94a3b8' }}, grid: {{ color: '#334155' }} }},
        y: {{ ticks: {{ color: '#94a3b8' }}, grid: {{ color: '#334155' }}, min: 0, max: 100 }}
      }}
    }}
  }});

  // Render Tabla CSV
  if (csvData.length > 0) {{
    const cols = Object.keys(csvData[0]);
    document.getElementById('csvHead').innerHTML = '<tr>' + cols.map(c => `<th>${{c}}</th>`).join('') + '</tr>';
    const cBody = document.getElementById('csvBody');
    csvData.forEach(row => {{
      const tr = document.createElement('tr');
      tr.innerHTML = cols.map(c => `<td>${{row[c]}}</td>`).join('');
      cBody.appendChild(tr);
    }});
  }} else {{
    document.getElementById('csvBody').innerHTML = '<tr><td>No se encontró /content/tabulador_costos_siniestros.csv</td></tr>';
  }}

  function filterTable() {{
    const q = document.getElementById('searchInput').value.toLowerCase();
    document.querySelectorAll('#csvBody tr').forEach(tr => {{
      tr.style.display = tr.textContent.toLowerCase().includes(q) ? '' : 'none';
    }});
  }}
</script>
</body>
</html>
"""

# Guardar también una copia física en /content por si deseas descargarla
with open("/content/dashboard_siniestros_real.html", "w", encoding="utf-8") as f:
    f.write(html_code)

# Renderizar directamente en la salida de la celda de Colab
display(HTML(html_code))

Run Name,Accuracy,F1-Score,Latencia,SLO


# Registros de MLFlow

In [18]:
import mlflow
import pandas as pd

def ver_experimentos_mlflow():
    print("🔍 Buscando experimentos en MLflow...\n")

    # Buscar todos los runs en todos los experimentos
    runs = mlflow.search_runs(search_all_experiments=True)

    if runs.empty:
        print("❌ No se encontraron experimentos en MLflow.")
        return

    # Seleccionar y renombrar las columnas más importantes para que sea legible
    columnas_interes = {
        'tags.mlflow.runName': 'Nombre del Run',
        'tags.tipo_incidente': 'Tipo Incidente',
        'tags.slo_status': 'Estado SLO',
        'metrics.accuracy': 'Accuracy',
        'metrics.f1_score': 'F1-Score',
        'metrics.latencia_ms': 'Latencia (ms)',
        'metrics.confianza_promedio': 'Confianza',
        'start_time': 'Fecha'
    }

    # Filtrar solo las columnas que existen en el DataFrame
    cols_presentes = [c for c in columnas_interes.keys() if c in runs.columns]
    df_resumen = runs[cols_presentes].copy()

    # Renombrar columnas
    df_resumen.rename(columns=columnas_interes, inplace=True)

    # Formatear la fecha
    if 'Fecha' in df_resumen.columns:
        df_resumen['Fecha'] = pd.to_datetime(df_resumen['Fecha']).dt.strftime('%Y-%m-%d %H:%M:%S')

    # Formatear métricas (opcional: redondear o multiplicar por 100)
    for col in ['Accuracy', 'F1-Score', 'Confianza']:
        if col in df_resumen.columns:
            df_resumen[col] = df_resumen[col].apply(lambda x: round(x * 100, 2) if pd.notnull(x) else None)

    if 'Latencia (ms)' in df_resumen.columns:
        df_resumen['Latencia (ms)'] = df_resumen['Latencia (ms)'].apply(lambda x: round(x, 2) if pd.notnull(x) else None)

    # Ordenar por fecha (los más recientes al final)
    if 'Fecha' in df_resumen.columns:
        df_resumen = df_resumen.sort_values('Fecha')

    # Rellenar vacíos con "-" para que se vea más limpio
    df_resumen = df_resumen.fillna("-")

    print(f"✅ Se encontraron {len(df_resumen)} ejecuciones registradas.\n")

    # Mostrar la tabla en Colab de forma bonita
    from IPython.display import display
    display(df_resumen)

# Ejecutar la función
ver_experimentos_mlflow()

🔍 Buscando experimentos en MLflow...

✅ Se encontraron 5 ejecuciones registradas.



,Nombre del Run,Tipo Incidente,Estado SLO,Accuracy,F1-Score,Latencia (ms),Confianza,Fecha
4,victorious-dog-712,-,-,-,-,-,-,2026-11-16 01:50:31
3,awesome-sloth-148,-,-,81.54,80.18,-,-,2026-11-16 01:59:54
2,monitoreo_produccion,-,-,95.64,95.29,15.97,-,2026-11-16 02:14:47
1,incidente_degradacion_modelo,degradacion_modelo,VIOLATION,-,-,-,-,2026-11-16 02:14:53
0,incidente_alta_latencia,alta_latencia,VIOLATION,95.64,95.29,2515.34,64.18,2026-11-16 02:15:43
